# NB-12: VACE Pipeline Integration -- Hint Pre-Computation and Additive Injection

## Learning Objectives
- Trace how VaceWanModel pre-computes all hints in a single forward pass before the DiT loop begins
- Understand how vace_patch_embedding tokenizes control signals separately from DiT's patch_embedding
- See the additive injection `x = x + hint * vace_scale` at matching even-indexed DiT layers

## Prerequisites
- **Prior notebooks:** NB-11 (VaceWanAttentionBlock accumulation pattern -- stack/unstack growing from `[2,B,S,D]` to `[4,B,S,D]` over 3 blocks), NB-07 (WanModel forward pass -- 8-step pipeline from input to output)
- **Assumed concepts:** VaceWanModel produces hints via the accumulation pattern (NB-11), DiT block loop (NB-07), `vace_layers_mapping` dictionary (NB-11)

## Concept Map
- **VACE hints** are computed once by VaceWanModel before the DiT loop starts, then injected per-layer during DiT forward
- This notebook shows the **pipeline-level** integration: how the orchestration code in `wan_video.py` connects VaceWanModel to DiT
- **Next:** Phase 5 (S2V Pipeline) covers the sketch-to-video path. Phase 6 (NB-18) shows how the pipeline dispatches between VACE and S2V

In [ ]:
# Source: diffsynth/models/wan_video_vace.py (setup)
# Source: diffsynth/models/wan_video_dit.py (setup)
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for VACE demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load DiT first (VACE depends on it)
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit_mod
_spec.loader.exec_module(dit_mod)

# Load VACE
_vace_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vace.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vace", _vace_path)
vace_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vace"] = vace_mod
_spec.loader.exec_module(vace_mod)

from diffsynth.models.wan_video_vace import VaceWanAttentionBlock, VaceWanModel
from diffsynth.models.wan_video_dit import DiTBlock, precompute_freqs_cis_3d
import torch

print("Setup complete.")

## VACE Pipeline Data Flow Diagram

```
VACE Pipeline Integration -- 2-Phase Data Flow
================================================

Phase A: Hint Pre-Computation (one-time, before DiT loop)
  Source: diffsynth/pipelines/wan_video.py, lines 1307-1312

  vace_context [list of tensors, each (vace_in_dim=16, F, H, W)]
       |
       v
  VaceWanModel.forward(x, vace_context, context, t_mod, freqs)
       |
       +-- Step A1: vace_patch_embedding: Conv3d(16, 384, (1,2,2), (1,2,2))
       |            per context item -> flatten -> pad to S_dit
       +-- Step A2: 3 VaceWanAttentionBlock passes (accumulation pattern, see NB-11)
       +-- Step A3: hints = unbind(stack)[:-1]  -> 3 hints, each [B, S, D]
       v
  vace_hints = [hint_0, hint_1, hint_2]

Phase B: Additive Injection (inside DiT block loop)
  Source: diffsynth/pipelines/wan_video.py, lines 1334, 1370-1375

  for block_id, block in enumerate(dit.blocks):  -- 5 blocks (0,1,2,3,4)
      x = block(x, context, t_mod, freqs)         -- standard DiT forward
      |
      +-- if block_id in vace_layers_mapping:      -- matches at 0, 2, 4
      |       hint_idx = mapping[block_id]
      |       x = x + vace_hints[hint_idx] * vace_scale
      |
      +-- else (block_id = 1, 3):                  -- no injection
              pass

  Layer 0:  x = block(x,...) + hint[0] * vace_scale   [INJECTED]
  Layer 1:  x = block(x,...)                            [no injection]
  Layer 2:  x = block(x,...) + hint[1] * vace_scale   [INJECTED]
  Layer 3:  x = block(x,...)                            [no injection]
  Layer 4:  x = block(x,...) + hint[2] * vace_scale   [INJECTED]
```

## Separate Patch Embeddings: VACE vs DiT

| Property | DiT (WanModel) | VACE (VaceWanModel) |
|----------|-----------------|---------------------|
| Layer | `patch_embedding` | `vace_patch_embedding` |
| Type | `Conv3d(in_dim, dim, patch_size, patch_size)` | `Conv3d(vace_in_dim, dim, patch_size, patch_size)` |
| Input channels | `in_dim=16` (noise latent) | `vace_in_dim=16` (control signal) |
| Output channels | `dim=384` | `dim=384` |
| Kernel/stride | `(1, 2, 2)` | `(1, 2, 2)` |

**Key:** Same architecture, different inputs. VACE does NOT share the DiT's patch embedding. Each has its own learned Conv3d weights, even though both project to the same model dimension `dim=384`.

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 27-51
# Source: diffsynth/pipelines/wan_video.py (simplified DiT block list)
dim, num_heads, ffn_dim = 384, 4, 1024
vace_in_dim = 16
vace_layers = (0, 2, 4)
patch_size = (1, 2, 2)
num_dit_layers = 5  # D-07: 5 DiT blocks so vace_layers (0,2,4) align at even indices

# VACE encoder
vace = VaceWanModel(
    vace_layers=vace_layers, vace_in_dim=vace_in_dim, patch_size=patch_size,
    has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim,
)
vace.eval()

# DiT blocks (standalone list, matching pipeline structure)
dit_blocks = torch.nn.ModuleList([
    DiTBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)
    for _ in range(num_dit_layers)
])
dit_blocks.eval()

vace_params = sum(p.numel() for p in vace.parameters())
dit_params = sum(p.numel() for p in dit_blocks.parameters())
print(f"VaceWanModel:  {vace_params:,} params  ({len(vace_layers)} blocks)")
print(f"DiT blocks:    {dit_params:,} params  ({num_dit_layers} blocks)")
print(f"VACE overhead: {vace_params / dit_params * 100:.1f}% of DiT")

assert len(vace.vace_blocks) == len(vace_layers), f"Expected {len(vace_layers)} VACE blocks"
assert len(dit_blocks) == num_dit_layers, f"Expected {num_dit_layers} DiT blocks"

In [ ]:
# Source: diffsynth/pipelines/wan_video.py (simplified inputs)
B = 1
F_vid, H_vid, W_vid = 4, 8, 8   # video spatial dims before patchify
S = F_vid * (H_vid // 2) * (W_vid // 2)  # after patchify with stride (1,2,2): 4 * 4 * 4 = 64
head_dim = dim // num_heads  # 96

# Build RoPE frequencies
# Source: diffsynth/models/wan_video_dit.py (precompute_freqs_cis_3d)
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)
F_p, H_p, W_p = F_vid, H_vid // 2, W_vid // 2  # post-patchify grid
freqs = torch.cat([
    f_freqs[:F_p].view(F_p, 1, 1, -1).expand(F_p, H_p, W_p, -1),
    h_freqs[:H_p].view(1, H_p, 1, -1).expand(F_p, H_p, W_p, -1),
    w_freqs[:W_p].view(1, 1, W_p, -1).expand(F_p, H_p, W_p, -1),
], dim=-1).reshape(F_p * H_p * W_p, 1, -1)
assert freqs.shape == torch.Size([S, 1, head_dim // 2])

# Fabricate tensors
x = torch.randn(B, S, dim)                           # noisy latent (already patchified)
context = torch.randn(B, 10, dim)                     # text embeddings
t_mod = torch.randn(B, 6, dim)                        # time modulation
vace_context = [torch.randn(vace_in_dim, F_vid, H_vid, W_vid)]  # single control signal

assert x.shape == torch.Size([B, S, dim])
assert context.shape == torch.Size([B, 10, dim])
assert t_mod.shape == torch.Size([B, 6, dim])
assert vace_context[0].shape == torch.Size([vace_in_dim, F_vid, H_vid, W_vid])

print(f"x (patchified latent):    {x.shape}")
print(f"context (text):           {context.shape}")
print(f"t_mod (time modulation):  {t_mod.shape}")
print(f"freqs (RoPE):             {freqs.shape}")
print(f"vace_context:             list of {len(vace_context)} tensor(s), shape {vace_context[0].shape}")

## Phase A: Hint Pre-Computation

Source: `diffsynth/pipelines/wan_video.py`, lines 1307-1312

Before the DiT block loop begins, all VACE hints are computed in a single forward pass. This is the key efficiency insight: hints depend only on the control signal and the *initial* noisy latent `x`, not on the DiT's intermediate states.

In [ ]:
# Source: diffsynth/models/wan_video_vace.py, lines 58-63
# Step A1: Patch-embed each context item

# Single context item: [vace_in_dim, F, H, W]
raw = vace_context[0]
print(f"Raw control signal:     {raw.shape}  (vace_in_dim={vace_in_dim}, F={F_vid}, H={H_vid}, W={W_vid})")

# Conv3d: (vace_in_dim, dim, patch_size, patch_size)
with torch.no_grad():
    embedded = vace.vace_patch_embedding(raw.unsqueeze(0))  # add batch dim
print(f"After vace_patch_embedding: {embedded.shape}  (Conv3d output)")
assert embedded.shape[0] == 1  # batch
assert embedded.shape[1] == dim  # output channels

# Flatten spatial dims and transpose to sequence format
embedded_flat = embedded.flatten(2).transpose(1, 2)
print(f"After flatten+transpose:    {embedded_flat.shape}  [1, S_vace, dim]")

# Pad to match DiT sequence length
S_vace = embedded_flat.size(1)
S_dit = x.shape[1]
if S_vace < S_dit:
    padding = embedded_flat.new_zeros(1, S_dit - S_vace, dim)
    embedded_padded = torch.cat([embedded_flat, padding], dim=1)
else:
    embedded_padded = embedded_flat
print(f"After padding to S_dit:     {embedded_padded.shape}  (S_vace={S_vace} -> S_dit={S_dit})")
assert embedded_padded.shape == torch.Size([1, S_dit, dim])

# Compare with DiT's patch embedding
print(f"\nVACE patch_embedding: Conv3d({vace_in_dim}, {dim}, {patch_size}, {patch_size})")
print(f"DiT  patch_embedding: Conv3d(16, {dim}, {patch_size}, {patch_size})")
print("Key: same kernel/stride, different input channels, separate weights")

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 1307-1312
# Pre-compute ALL hints in one forward pass
print("Step A2-A3: VaceWanModel.forward() -- full hint pre-computation")
print("=" * 60)

with torch.no_grad():
    vace_hints = vace(x, vace_context, context, t_mod, freqs)

assert len(vace_hints) == len(vace_layers), f"Expected {len(vace_layers)} hints, got {len(vace_hints)}"
print(f"\nPre-computed {len(vace_hints)} hints:")
for i, h in enumerate(vace_hints):
    assert h.shape == torch.Size([B, S, dim])
    print(f"  vace_hints[{i}]: {h.shape}  (for DiT layer {vace_layers[i]})")

print(f"\nvace_layers_mapping: {vace.vace_layers_mapping}")
print("Maps: DiT block_id -> VACE hint index")
assert vace.vace_layers_mapping == {0: 0, 2: 1, 4: 2}

## Phase B: Additive Injection in DiT Block Loop

Source: `diffsynth/pipelines/wan_video.py`, lines 1334, 1370-1375

During inference, the pre-computed hints are injected additively after matching DiT blocks. The injection is a simple element-wise addition scaled by `vace_scale`:

```python
x = x + current_vace_hint * vace_scale
```

Only layers at even indices (0, 2, 4) receive hints. Layers 1 and 3 run without modification.

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, lines 1334, 1367, 1370-1375
# Simplified: skip gradient checkpointing, sequence parallelism, VAP, TeaCache
print("Phase B: DiT block loop with VACE injection")
print("=" * 60)

vace_scale = 1.0
x_running = x.clone()  # copy so we can compare with/without VACE later
injection_count = 0

with torch.no_grad():
    for block_id, block in enumerate(dit_blocks):
        # Standard DiT forward (line 1367)
        x_running = block(x_running, context, t_mod, freqs)

        # VACE injection (lines 1370-1375)
        if block_id in vace.vace_layers_mapping:
            hint_idx = vace.vace_layers_mapping[block_id]
            current_vace_hint = vace_hints[hint_idx]
            assert current_vace_hint.shape == torch.Size([B, S, dim]), f"Hint shape mismatch at layer {block_id}"
            x_running = x_running + current_vace_hint * vace_scale
            injection_count += 1
            print(f"  Layer {block_id}: DiT forward + VACE injection (hint[{hint_idx}], scale={vace_scale})")
            print(f"           x shape: {x_running.shape}")
        else:
            assert block_id not in vace.vace_layers_mapping, f"Layer {block_id} should NOT be in mapping"
            print(f"  Layer {block_id}: DiT forward only (no VACE injection)")
            print(f"           x shape: {x_running.shape}")

        assert x_running.shape == torch.Size([B, S, dim])

print(f"\nFinal x: {x_running.shape}")
print(f"Injections: layers {[l for l in range(num_dit_layers) if l in vace.vace_layers_mapping]}")
print(f"No injection: layers {[l for l in range(num_dit_layers) if l not in vace.vace_layers_mapping]}")

assert x_running.shape == torch.Size([B, S, dim])
assert injection_count == len(vace_layers), f"Expected {len(vace_layers)} injections, got {injection_count}"
injected_layers = [l for l in range(num_dit_layers) if l in vace.vace_layers_mapping]
assert injected_layers == [0, 2, 4], f"Expected injection at [0,2,4], got {injected_layers}"

In [ ]:
# Source: demonstrating vace_scale effect (pedagogical demo)
# Run the same DiT blocks WITHOUT VACE injection for comparison
x_no_vace = x.clone()

with torch.no_grad():
    for block_id, block in enumerate(dit_blocks):
        x_no_vace = block(x_no_vace, context, t_mod, freqs)

# Compare
diff = (x_running - x_no_vace).abs()
print(f"With VACE:    {x_running.shape}  mean={x_running.mean():.4f}")
print(f"Without VACE: {x_no_vace.shape}  mean={x_no_vace.mean():.4f}")
print(f"Difference:   mean_abs_diff={diff.mean():.4f}, max_abs_diff={diff.max():.4f}")
assert diff.mean() > 0, "VACE injection should change the output"
assert diff.max() > 0, "Max difference should also be non-zero"
assert x_no_vace.shape == torch.Size([B, S, dim])
print("\nVACE injection produces a measurably different output from the uncontrolled path.")

In [ ]:
# Source: diffsynth/pipelines/wan_video.py, line 1375
# Demonstrate vace_scale effect: x = x + hint * vace_scale
print("vace_scale effect on injection strength")
print("=" * 60)

prev_diff = -1.0
for scale in [0.0, 0.5, 1.0, 2.0]:
    x_scaled = x.clone()
    with torch.no_grad():
        for block_id, block in enumerate(dit_blocks):
            x_scaled = block(x_scaled, context, t_mod, freqs)
            if block_id in vace.vace_layers_mapping:
                hint_idx = vace.vace_layers_mapping[block_id]
                x_scaled = x_scaled + vace_hints[hint_idx] * scale

    diff_from_base = (x_scaled - x_no_vace).abs().mean().item()
    print(f"  vace_scale={scale:.1f}:  mean_diff_from_no_vace={diff_from_base:.4f}")
    assert x_scaled.shape == torch.Size([B, S, dim])
    assert diff_from_base >= prev_diff, f"Diff should increase with scale: {diff_from_base} < {prev_diff}"
    prev_diff = diff_from_base

# vace_scale=0.0 should match no-VACE exactly
x_scale0 = x.clone()
with torch.no_grad():
    for block_id, block in enumerate(dit_blocks):
        x_scale0 = block(x_scale0, context, t_mod, freqs)
        if block_id in vace.vace_layers_mapping:
            hint_idx = vace.vace_layers_mapping[block_id]
            x_scale0 = x_scale0 + vace_hints[hint_idx] * 0.0

assert torch.allclose(x_scale0, x_no_vace, atol=1e-6), "vace_scale=0 must match no-VACE output"
assert x_scale0.shape == torch.Size([B, S, dim])
print("\nVerified: vace_scale=0.0 produces identical output to no-VACE path.")

### Shape Trace Summary

| Step | Operation | Input Shape | Output Shape | Source |
|------|-----------|-------------|--------------|--------|
| A1 | `vace_patch_embedding` (Conv3d) | `[1, 16, 4, 8, 8]` | `[1, 384, 4, 4, 4]` | `wan_video_vace.py`, line 58 |
| A1 | flatten + transpose | `[1, 384, 4, 4, 4]` | `[1, 64, 384]` | `wan_video_vace.py`, line 59 |
| A1 | pad to S_dit | `[1, 64, 384]` | `[1, 64, 384]` | `wan_video_vace.py`, lines 60-63 |
| A2 | 3x VaceWanAttentionBlock | `[1, 64, 384]` | `[4, 1, 64, 384]` | `wan_video_vace.py`, line 85 |
| A3 | `unbind(c)[:-1]` | `[4, 1, 64, 384]` | `3x [1, 64, 384]` | `wan_video_vace.py`, line 86 |
| B | DiT block + VACE injection | `[1, 64, 384]` | `[1, 64, 384]` | `wan_video.py`, line 1375 |

**Key shapes:**
- Control signal: `[vace_in_dim=16, F=4, H=8, W=8]` -- raw input before patch embedding
- After patchify: `[1, S=64, dim=384]` -- same sequence length as DiT tokens
- Each hint: `[B=1, S=64, dim=384]` -- identical shape to DiT's `x`, enabling element-wise addition
- `x = x + hint * vace_scale` preserves shape `[1, 64, 384]` throughout the entire DiT block loop

## Key Architectural Insights

1. **Pre-compute once, inject many:** VACE hints are computed once before the DiT loop. They do not depend on DiT intermediate states, so there is no sequential dependency between VACE and DiT blocks. This is much lighter than ControlNet which requires per-layer computation.

2. **Additive injection is simple:** `x = x + hint * vace_scale` is a simple element-wise addition. No extra attention, no gating, no concatenation. The hint has the same shape `[B, S, D]` as `x`.

3. **Even-layer alignment:** VACE injects at even DiT layers (0, 2, 4). This means every other DiT block gets control signal influence, while the intervening blocks process freely. In production (30 DiT layers, 15 VACE blocks), this creates a regular interleaving pattern.

4. **vace_scale as control strength:** Setting `vace_scale=0` completely disables VACE control. Values >1.0 amplify control. This allows runtime control strength tuning without recomputing hints.

## Summary

### Key Takeaways
- **VACE hints are pre-computed in one forward pass**, separate from the DiT loop. VaceWanModel processes the control signal through its own patch embedding and attention blocks, producing one hint per VACE block (3 hints for 3 blocks).
- **Additive injection** `x = x + hint * vace_scale` at matching even DiT layers (0, 2, 4) is the entire integration mechanism. No extra attention layers, no gating, no concatenation.
- **vace_layers_mapping** provides O(1) lookup from DiT `block_id` to hint index: `{0: 0, 2: 1, 4: 2}`. This dictionary is constructed once at model init time.

### Source References

| Symbol | Location |
|--------|----------|
| VaceWanModel.forward | `diffsynth/models/wan_video_vace.py`, line 53 |
| vace_patch_embedding | `diffsynth/models/wan_video_vace.py`, line 51 |
| VACE hint pre-computation | `diffsynth/pipelines/wan_video.py`, line 1307 |
| DiT block loop | `diffsynth/pipelines/wan_video.py`, line 1334 |
| VACE additive injection | `diffsynth/pipelines/wan_video.py`, line 1375 |
| vace_scale usage | `diffsynth/pipelines/wan_video.py`, line 1375 |

### Next
Phase 5 (S2V Pipeline) covers the sketch-to-video path with FramePack motion injection and audio conditioning. Phase 6 (Integration) shows how the pipeline dispatches between VACE and S2V.

## Exercises

### Exercise 1 -- What happens with consecutive vace_layers?
What happens if you make `vace_layers=(0, 1, 2)` instead of `(0, 2, 4)`? How does the injection pattern change? Try modifying `vace_layers` and re-running the loop. Which DiT layers now receive hints, and which run freely? Does the even/odd interleaving pattern still hold?

### Exercise 2 -- Production scale
In production, WAN 2.2 uses 30 DiT layers and 15 VACE blocks at even indices (0, 2, ..., 28). How many DiT layers receive NO VACE injection? What percentage of DiT compute is "uncontrolled"? (Answer: 15 out of 30 layers = 50% uncontrolled.)

### Exercise 3 -- Multiple control signals
The current demo uses a single control signal (`len(vace_context)==1`). What happens to the initial `c` tensor shape if you provide 2 control signals? (Hint: look at the `torch.cat` in `VaceWanModel.forward` -- it concatenates along dim=0, so `c` goes from `[1, S, D]` to `[2, S, D]`.) Try it: `vace_context = [torch.randn(16, 4, 8, 8), torch.randn(16, 4, 8, 8)]` and trace the shapes.